In [ ]:
#############################  
daily_steps = """
DAILY RUN STEPS

1. Restart the notebook kernel.

2. Run this main trading cell.
   Confirm:
   - Login Successful
   - TRADING SYSTEM STARTED
   - Running on http://127.0.0.1:5000

3. Open PowerShell and run:
   ngrok http 5000

4. Copy the HTTPS ngrok URL and add /webhook.
   Example:
   https://abcd.ngrok-free.app/webhook

5. Paste that full URL in Chartink webhook URL.

6. Use this Chartink webhook body:

{
  "stocks": "{{stocks}}",
  "trigger_prices": "{{trigger_prices}}",
  "triggered_at": "{{triggered_at}}",
  "scan_name": "{{scan_name}}",
  "scan_url": "{{scan_url}}",
  "alert_name": "{{alert_name}}",
  "webhook_url": "{{webhook_url}}"
}

7. Keep both running:
   - This notebook cell
   - ngrok terminal

8. Do NOT run Webhook_file.py during live trading.
"""
############################################################

In [ ]:
# WOrking

# ==================== SIMPLIFIED TRADING SYSTEM ====================

import csv
import json
import logging
import os
import threading
import time
from datetime import datetime
from dateutil.relativedelta import relativedelta
import pandas as pd
import pyotp
import yaml
from NorenRestApiPy.NorenApi import NorenApi
from flask import Flask, request


# ==================== EXECUTION SETUP ====================
execution_folder = r'D:\AlgoRepo\ShoonyaAPI_Code\Testing_Use\Execution'
os.makedirs(execution_folder, exist_ok=True)

class ExecutionLogger:
    """Execution logging class"""
    def __init__(self, folder_path):
        self.folder = folder_path
        self.session_id = datetime.now().strftime('%Y%m%d_%H%M%S')
        self.session_file = os.path.join(folder_path, f'session_{self.session_id}.json')
        self.trades_log = []
        self.webhooks_log = []
        
    def log_webhook(self, webhook_data, filtered_count, parsed_stocks=None):
        """Log incoming webhook"""
        parsed_stocks = parsed_stocks or []
        entry = {
            'timestamp': datetime.now().strftime('%Y-%m-%d %H:%M:%S'),
            'scan_name': webhook_data.get('scan_name', ''),
            'alert_name': webhook_data.get('alert_name', ''),
            'triggered_at': webhook_data.get('triggered_at', ''),
            'total_stocks': len(parsed_stocks),
            'filtered_stocks': filtered_count,
            'stocks': parsed_stocks,
            'raw_payload': webhook_data
        }
        self.webhooks_log.append(entry)
        self._save()
            
    def log_trade(self, symbol, order_id, entry_price, quantity, sl, tp):
        """Log new trade"""
        entry = {
            'timestamp': datetime.now().strftime('%Y-%m-%d %H:%M:%S'),
            'symbol': symbol,
            'order_id': order_id,
            'entry_price': entry_price,
            'quantity': quantity,
            'sl_price': sl,
            'tp_price': tp,
            'status': 'ACTIVE'
        }
        self.trades_log.append(entry)
        self._save()
        
    def log_exit(self, order_id, exit_price, exit_reason):
        """Log trade exit"""
        for trade in self.trades_log:
            if trade['order_id'] == order_id:
                trade['status'] = 'CLOSED'
                trade['exit_price'] = exit_price
                trade['exit_reason'] = exit_reason
                trade['exit_time'] = datetime.now().strftime('%Y-%m-%d %H:%M:%S')
                trade['pnl'] = (trade['entry_price'] - exit_price) * trade['quantity']
                self._save()
                break
    
    def _save(self):
        """Save logs to JSON"""
        data = {
            'session_id': self.session_id,
            'created_at': datetime.now().strftime('%Y-%m-%d %H:%M:%S'),
            'webhooks': self.webhooks_log,
            'trades': self.trades_log
        }
        with open(self.session_file, 'w') as f:
            json.dump(data, f, indent=2)


# Initialize logger
logger = ExecutionLogger(execution_folder)

# ==================== LOGIN ====================
class ShoonyaApiPy(NorenApi):
    def __init__(self):
        super().__init__(host='https://trade.shoonya.com/NorenWClientWeb/', 
                        websocket='wss://trade.shoonya.com/NorenWSWeb/')

api = ShoonyaApiPy()

with open('cred.yml') as f:
    cred = yaml.load(f, Loader=yaml.FullLoader)

SuperToken = "5ddcb928a35c7f1e240427e652039b2700b5d972759e11827b7cfb88ce88735a"
userId = cred['user']
Passwrd = cred['pwd']

print("=" * 80)
print("🔐 LOGGING INTO SHOONYA BROKER...")
print("=" * 80)

ret = api.set_session(userId, Passwrd, SuperToken)

if ret:
    print("✅ Login Successful!")
    print(f"   User: {userId}")
else:
    print("❌ Login Failed!")
    exit()

# ==================== FLASK APP ====================
app = Flask(__name__)
logging.basicConfig(level=logging.INFO)

active_trades = {}
lock = threading.Lock()

VALID_TIME_WINDOWS = [
    ("09:45", "10:00"),
    ("10:15", "10:30"),
    ("10:30", "10:45"),
    ("10:45", "11:00"),
    ("13:00", "13:15")
]

MAX_STOCKS_PER_TIME = 3
TRADING_CAP_PER_STOCK = 20000


def is_time_in_valid_window(time_str):
    """Check if time is in valid window"""
    check_time = datetime.strptime(time_str, "%H:%M").time()
    for start_str, end_str in VALID_TIME_WINDOWS:
        start_time = datetime.strptime(start_str, "%H:%M").time()
        end_time = datetime.strptime(end_str, "%H:%M").time()
        if start_time <= check_time <= end_time:
            return True
    return False


def parse_webhook_payload(data):
    """Parse webhook payload"""
    if not data:
        raise ValueError("Webhook payload is empty")

    stocks_raw = data.get("stocks", "")
    prices_raw = data.get("trigger_prices", "")
    triggered_at = data.get("triggered_at", "")

    if not stocks_raw or not prices_raw or not triggered_at:
        raise ValueError("Missing required fields in webhook")

    symbols = [s.strip() for s in stocks_raw.split(",") if s.strip()]
    prices = [p.strip() for p in prices_raw.split(",") if p.strip()]

    if len(symbols) != len(prices):
        raise ValueError(f"Stocks count and price count mismatch")

    trigger_time = datetime.strptime(triggered_at.strip().upper(), "%I:%M %p").strftime("%H:%M")

    parsed_stocks = []
    for symbol, price in zip(symbols, prices):
        parsed_stocks.append({
            "symbol": symbol,
            "trigger_price": float(price),
            "time": trigger_time,
            "scan_name": data.get("scan_name", ""),
            "scan_url": data.get("scan_url", ""),
            "alert_name": data.get("alert_name", "")
        })

    return parsed_stocks


def filter_stocks(stocks):
    """Filter stocks by time window and count"""
    if not stocks:
        print("⏭️  No stocks received")
        return []

    if len(stocks) > MAX_STOCKS_PER_TIME:
        print(f"⏭️  Received {len(stocks)} stocks (max {MAX_STOCKS_PER_TIME}). Skipping.")
        return []

    for stock in stocks:
        time = stock.get('time', '').strip()
        if not is_time_in_valid_window(time):
            print(f"⏭️  Time {time} not in valid windows. Skipping.")
            return []

    print(f"✅ Filtered {len(stocks)} stocks")
    return stocks


def place_sell_orders(stocks):
    """Place SELL orders immediately for filtered stocks"""
    print(f"\n📥 PLACING SELL ORDERS:")
    print("-" * 80)
    
    for stock in stocks:
        symbol = stock.get('symbol', 'UNKNOWN').strip()
        entry_time = stock.get('time', '')
        symbol_eq = f"{symbol}-EQ"
        
        try:
            # Get current price
            quote = api.get_quotes(exchange='NSE', token=symbol_eq)
            current_ltp = float(quote.get("lp", 0))
            
            if current_ltp <= 0:
                print(f"❌ {symbol_eq}: Invalid price")
                continue
            
            quantity = int(TRADING_CAP_PER_STOCK / current_ltp)
            
            # Place SELL order
            response = api.place_order(
                buy_or_sell='S',
                product_type='I',
                exchange='NSE',
                tradingsymbol=symbol_eq,
                quantity=quantity,
                discloseqty=0,
                price_type='MKT',
                retention='DAY',
                remarks='Webhook_Sell'
            )
            
            order_id = response.get('norenordno')
            
            if order_id:
                sl_price = current_ltp * 1.006
                tp_price = current_ltp * 0.992
                
                with lock:
                    active_trades[order_id] = {
                        'symbol': symbol_eq,
                        'entry_time': entry_time,
                        'entry_price': current_ltp,
                        'quantity': quantity,
                        'status': 'ACTIVE',
                        'order_id': order_id,
                        'sl_price': sl_price,
                        'tp_price': tp_price,
                        'placed_at': datetime.now()
                    }
                
                logger.log_trade(symbol_eq, order_id, current_ltp, quantity, sl_price, tp_price)
                
                print(f"✅ {symbol_eq} | SELL @ {current_ltp:.2f} | Qty: {quantity}")
                print(f"   Order ID: {order_id} | TP: {tp_price:.2f} | SL: {sl_price:.2f}")
            else:
                print(f"❌ {symbol_eq}: Order failed - {response.get('emsg', 'Unknown error')}")
        
        except Exception as e:
            print(f"❌ {symbol}: {str(e)[:50]}")


@app.route('/webhook', methods=['POST'])
def webhook_handler():
    """Webhook endpoint"""
    try:
        data = request.json
        print(f"\n{'='*80}")
        print(f"📡 WEBHOOK RECEIVED - {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
        print(f"{'='*80}")

        parsed_stocks = parse_webhook_payload(data)
        logger.log_webhook(data, 0, parsed_stocks)

        filtered_stocks = filter_stocks(parsed_stocks)
        logger.webhooks_log[-1]['filtered_stocks'] = len(filtered_stocks)
        logger._save()

        if not filtered_stocks:
            print("⏭️  No stocks passed filter")
            return {"status": "no_match", "received": len(parsed_stocks), "filtered": 0}, 200

        place_sell_orders(filtered_stocks)

        return {"status": "success", "received": len(parsed_stocks), "placed": len(filtered_stocks)}, 200

    except Exception as e:
        print(f"❌ Webhook error: {e}")
        return {"status": "error", "message": str(e)}, 400


def monitor_trades():
    """Monitor trades every 30 seconds"""
    iteration = 0
    
    while True:
        try:
            iteration += 1
            
            with lock:
                if not active_trades:
                    print(f"⏳ [{iteration}] Waiting for trades...")
                    time.sleep(30)
                    continue
                
                print(f"\n{'='*80}")
                print(f"📊 MONITOR CHECK #{iteration} - {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
                print(f"Active trades: {len(active_trades)}")
                print('='*80)
                
                closed_orders = []
                
                for order_id, trade in list(active_trades.items()):
                    if trade['status'] != 'ACTIVE':
                        continue
                    
                    symbol = trade['symbol']
                    entry_price = trade['entry_price']
                    quantity = trade['quantity']
                    sl_price = trade['sl_price']
                    tp_price = trade['tp_price']
                    
                    try:
                        quote = api.get_quotes(exchange='NSE', token=symbol)
                        current_ltp = float(quote.get("lp", entry_price))
                        
                        print(f"\n{symbol} | Entry: {entry_price:.2f} | Current: {current_ltp:.2f}")
                        print(f"  TP: {tp_price:.2f} | SL: {sl_price:.2f}")
                        
                        # Check TP hit
                        if current_ltp <= tp_price:
                            print(f"✅ TP HIT! Closing...")
                            close_response = api.place_order(
                                buy_or_sell='B',
                                product_type='I',
                                exchange='NSE',
                                tradingsymbol=symbol,
                                quantity=quantity,
                                discloseqty=0,
                                price_type='MKT',
                                retention='DAY',
                                remarks='Close_TP'
                            )
                            
                            if close_response.get('norenordno'):
                                pnl = (entry_price - current_ltp) * quantity
                                print(f"💰 P&L: {pnl:.2f}")
                                trade['status'] = 'CLOSED_TP'
                                logger.log_exit(order_id, current_ltp, 'TP_HIT')
                                closed_orders.append(order_id)
                        
                        # Check SL hit
                        elif current_ltp >= sl_price:
                            print(f"❌ SL HIT! Closing...")
                            close_response = api.place_order(
                                buy_or_sell='B',
                                product_type='I',
                                exchange='NSE',
                                tradingsymbol=symbol,
                                quantity=quantity,
                                discloseqty=0,
                                price_type='MKT',
                                retention='DAY',
                                remarks='Close_SL'
                            )
                            
                            if close_response.get('norenordno'):
                                pnl = (entry_price - current_ltp) * quantity
                                print(f"💔 P&L: {pnl:.2f}")
                                trade['status'] = 'CLOSED_SL'
                                logger.log_exit(order_id, current_ltp, 'SL_HIT')
                                closed_orders.append(order_id)
                        else:
                            pnl = (entry_price - current_ltp) * quantity
                            print(f"📈 P&L: {pnl:.2f}")
                    
                    except Exception as e:
                        print(f"⚠️  Error checking {symbol}: {str(e)[:50]}")
                
                for order_id in closed_orders:
                    del active_trades[order_id]
            
            time.sleep(30)
        
        except Exception as e:
            print(f"❌ Monitor error: {e}")
            time.sleep(30)


# ==================== START SYSTEM ====================
print("\n" + "="*80)
print("🚀 TRADING SYSTEM STARTED - SIMPLIFIED")
print("="*80)
print(f"📝 Session ID: {logger.session_id}")
print(f"📋 Log file: {logger.session_file}")
print("="*80 + "\n")

monitor_thread = threading.Thread(target=monitor_trades, daemon=True)
monitor_thread.start()
print("✅ Monitor thread started")

print("⏳ Starting webhook server on port 5000...")
app.run(host='0.0.0.0', port=5000, debug=False)

🔐 LOGGING INTO SHOONYA BROKER...
✅ Login Successful!
   User: FA74468

🚀 TRADING SYSTEM STARTED - SIMPLIFIED
📝 Session ID: 20260504_210916
📋 Log file: D:\AlgoRepo\ShoonyaAPI_Code\Testing_Use\Execution\session_20260504_210916.json

⏳ [1] Waiting for trades...
✅ Monitor thread started
⏳ Starting webhook server on port 5000...
 * Serving Flask app '__main__'
 * Debug mode: off


INFO:werkzeug:WARNING: This is a development server. Do not use it in a production deployment. Use a production WSGI server instead.
 * Running on all addresses (0.0.0.0)
 * Running on http://127.0.0.1:5000
 * Running on http://192.168.1.6:5000
INFO:werkzeug:Press CTRL+C to quit


⏳ [2] Waiting for trades...
⏳ [3] Waiting for trades...
⏳ [4] Waiting for trades...
⏳ [5] Waiting for trades...
⏳ [6] Waiting for trades...
⏳ [7] Waiting for trades...
⏳ [8] Waiting for trades...
⏳ [9] Waiting for trades...
⏳ [10] Waiting for trades...
⏳ [11] Waiting for trades...
⏳ [12] Waiting for trades...
⏳ [13] Waiting for trades...
⏳ [14] Waiting for trades...
⏳ [15] Waiting for trades...
⏳ [16] Waiting for trades...
⏳ [17] Waiting for trades...
⏳ [18] Waiting for trades...
⏳ [19] Waiting for trades...
⏳ [20] Waiting for trades...
⏳ [21] Waiting for trades...
⏳ [22] Waiting for trades...
⏳ [23] Waiting for trades...
⏳ [24] Waiting for trades...
⏳ [25] Waiting for trades...
⏳ [26] Waiting for trades...
⏳ [27] Waiting for trades...
⏳ [28] Waiting for trades...
⏳ [29] Waiting for trades...
⏳ [30] Waiting for trades...
⏳ [31] Waiting for trades...
⏳ [32] Waiting for trades...
⏳ [33] Waiting for trades...
⏳ [34] Waiting for trades...
⏳ [35] Waiting for trades...
⏳ [36] Waiting for tra

INFO:werkzeug:127.0.0.1 - - [04/May/2026 21:50:35] "POST /webhook HTTP/1.1" 200 -



📡 WEBHOOK RECEIVED - 2026-05-04 21:50:35
⏭️  Received 7 stocks (max 3). Skipping.
⏭️  No stocks passed filter
⏳ [84] Waiting for trades...
⏳ [85] Waiting for trades...
⏳ [86] Waiting for trades...


...

INFO:werkzeug:127.0.0.1 - - [04/May/2026 21:53:44] "POST /webhook HTTP/1.1" 200 -


✅ ASTEC-EQ | SELL @ 712.60 | Qty: 28
   Order ID: 26050400454383 | TP: 706.90 | SL: 716.88

📊 MONITOR CHECK #90 - 2026-05-04 21:53:47
Active trades: 2

ASTEC-EQ | Entry: 712.60 | Current: 712.60
  TP: 706.90 | SL: 716.88
📈 P&L: 0.00

ASTEC-EQ | Entry: 712.60 | Current: 712.60
  TP: 706.90 | SL: 716.88
📈 P&L: 0.00

📊 MONITOR CHECK #91 - 2026-05-04 21:54:17
Active trades: 2

ASTEC-EQ | Entry: 712.60 | Current: 712.60
  TP: 706.90 | SL: 716.88
📈 P&L: 0.00

ASTEC-EQ | Entry: 712.60 | Current: 712.60
  TP: 706.90 | SL: 716.88
📈 P&L: 0.00

📊 MONITOR CHECK #92 - 2026-05-04 21:54:47
Active trades: 2

ASTEC-EQ | Entry: 712.60 | Current: 712.60
  TP: 706.90 | SL: 716.88
📈 P&L: 0.00

ASTEC-EQ | Entry: 712.60 | Current: 712.60
  TP: 706.90 | SL: 716.88
📈 P&L: 0.00

📊 MONITOR CHECK #93 - 2026-05-04 21:55:17
Active trades: 2

ASTEC-EQ | Entry: 712.60 | Current: 712.60
  TP: 706.90 | SL: 716.88
📈 P&L: 0.00

ASTEC-EQ | Entry: 712.60 | Current: 712.60
  TP: 706.90 | SL: 716.88
📈 P&L: 0.00

📡 WEBHOOK RE

INFO:werkzeug:127.0.0.1 - - [04/May/2026 21:55:26] "POST /webhook HTTP/1.1" 200 -


✅ ASTEC-EQ | SELL @ 712.60 | Qty: 28
   Order ID: 26050400454426 | TP: 706.90 | SL: 716.88

📊 MONITOR CHECK #94 - 2026-05-04 21:55:48
Active trades: 4

ASTEC-EQ | Entry: 712.60 | Current: 712.60
  TP: 706.90 | SL: 716.88
📈 P&L: 0.00

ASTEC-EQ | Entry: 712.60 | Current: 712.60
  TP: 706.90 | SL: 716.88
📈 P&L: 0.00

WIPRO-EQ | Entry: 200.75 | Current: 200.75
  TP: 199.14 | SL: 201.95
📈 P&L: 0.00

ASTEC-EQ | Entry: 712.60 | Current: 712.60
  TP: 706.90 | SL: 716.88
📈 P&L: 0.00

📊 MONITOR CHECK #95 - 2026-05-04 21:56:18
Active trades: 4

ASTEC-EQ | Entry: 712.60 | Current: 712.60
  TP: 706.90 | SL: 716.88
📈 P&L: 0.00

ASTEC-EQ | Entry: 712.60 | Current: 712.60
  TP: 706.90 | SL: 716.88
📈 P&L: 0.00

WIPRO-EQ | Entry: 200.75 | Current: 200.75
  TP: 199.14 | SL: 201.95
📈 P&L: 0.00

ASTEC-EQ | Entry: 712.60 | Current: 712.60
  TP: 706.90 | SL: 716.88
📈 P&L: 0.00

📊 MONITOR CHECK #96 - 2026-05-04 21:56:48
Active trades: 4

ASTEC-EQ | Entry: 712.60 | Current: 712.60
  TP: 706.90 | SL: 716.88
📈 P


📊 MONITOR CHECK #102 - 2026-05-04 21:59:50
Active trades: 4

ASTEC-EQ | Entry: 712.60 | Current: 712.60
  TP: 706.90 | SL: 716.88
📈 P&L: 0.00

ASTEC-EQ | Entry: 712.60 | Current: 712.60
  TP: 706.90 | SL: 716.88
📈 P&L: 0.00

WIPRO-EQ | Entry: 200.75 | Current: 200.75
  TP: 199.14 | SL: 201.95
📈 P&L: 0.00

ASTEC-EQ | Entry: 712.60 | Current: 712.60
  TP: 706.90 | SL: 716.88
📈 P&L: 0.00


# Trading System Flow Chart

## Overall Flow

```
┌─────────────────────────────────────────────────────────────────┐
│                    TRADING SYSTEM WORKFLOW                       │
└─────────────────────────────────────────────────────────────────┘

                              ↓
                    
┌─────────────────────────────────────────────────────────────────┐
│  1️⃣  WEBHOOK RECEIVES STOCKS                                    │
│  ✓ Chartink sends: SEPOWER, ASTEC, EDUCOMP                      │
│  ✓ With prices and time triggered                               │
└─────────────────────────────────────────────────────────────────┘

                              ↓
                    
┌─────────────────────────────────────────────────────────────────┐
│  2️⃣  PARSE & FILTER                                             │
│  ✓ Parse JSON payload                                           │
│  ✓ Check time is in valid window (09:45-10:00, etc)            │
│  ✓ Check max 3 stocks per time                                  │
│  ✓ If any check fails → SKIP ENTIRE WEBHOOK                    │
└─────────────────────────────────────────────────────────────────┘

                              ↓
                    
┌─────────────────────────────────────────────────────────────────┐
│  3️⃣  PLACE SELL ORDERS (IMMEDIATELY)                            │
│  ✓ For each filtered stock:                                     │
│    - Get current LTP from NSE                                   │
│    - Calculate quantity = 20000 / LTP                           │
│    - Place SELL (SHORT) order at market price                   │
│  ✓ If order placed → Add to active_trades dictionary           │
│  ✓ Calculate TP = entry_price × 0.992 (profit target)          │
│  ✓ Calculate SL = entry_price × 1.006 (stop loss)              │
└─────────────────────────────────────────────────────────────────┘

                              ↓
                    
┌─────────────────────────────────────────────────────────────────┐
│  4️⃣  MONITOR LOOP (EVERY 30 SECONDS)                            │
│  ✓ Check each order in active_trades                           │
│  ✓ Get current LTP from NSE                                    │
│  ✓ Check 2 conditions:                                         │
│                                                                 │
│    IF current_ltp ≤ TP_PRICE                                   │
│    └─→ PROFIT TARGET HIT! Close at profit                      │
│                                                                 │
│    ELSE IF current_ltp ≥ SL_PRICE                              │
│    └─→ STOP LOSS HIT! Close at loss                            │
│                                                                 │
│    ELSE                                                         │
│    └─→ Keep waiting (print P&L)                                │
│                                                                 │
└─────────────────────────────────────────────────────────────────┘

                              ↓
                    
┌─────────────────────────────────────────────────────────────────┐
│  5️⃣  CLOSE ORDER (BUY to close SHORT)                           │
│  ✓ Place BUY order at market price                              │
│  ✓ If successful → Mark trade as CLOSED                        │
│  ✓ Log P&L = (entry_price - current_ltp) × quantity            │
│  ✓ Remove from active_trades                                   │
└─────────────────────────────────────────────────────────────────┘

                              ↓
                    
┌─────────────────────────────────────────────────────────────────┐
│  6️⃣  LOG TO JSON                                                │
│  ✓ Session file: session_YYYYMMDD_HHMMSS.json                  │
│  ✓ Records: webhooks received, trades opened, trades closed    │
│  ✓ Data: symbol, entry price, exit price, P&L, reason          │
└─────────────────────────────────────────────────────────────────┘
```

---

## Detailed Flow (With Decisions)

```
                        START
                          ↓
                    
        ┌─────────────────────────────────┐
        │  FLASK SERVER LISTENING          │
        │  Port 5000 /webhook              │
        └─────────────────────────────────┘
                          ↓
                    
        ┌─────────────────────────────────┐
        │  WEBHOOK RECEIVED                │
        │  Webhook_data = {                │
        │    stocks: "SEPOWER,ASTEC"       │
        │    trigger_prices: "3.75,541.8"  │
        │    triggered_at: "2:34 pm"       │
        │  }                               │
        └─────────────────────────────────┘
                          ↓
                    
        ┌─────────────────────────────────┐
        │  PARSE PAYLOAD                   │
        │  → symbols = [SEPOWER, ASTEC]   │
        │  → prices = [3.75, 541.8]       │
        │  → time = "14:34"               │
        └─────────────────────────────────┘
                          ↓
                    
        ┌─────────────────────────────────┐
        │  FILTER: Check Time Valid?       │
        │  14:34 in [09:45-10:00]?        │
        └──────┬──────────────────────────┘
               │
        ┌──────┴──────┐
        ↓             ↓
      NO            YES
      │              │
      X              ↓
   SKIP         ┌─────────────────────┐
            │  FILTER: Check Count    │
            │  2 stocks ≤ 3 max?     │
            └──────┬──────────────────┘
                   │
            ┌──────┴──────┐
            ↓             ↓
          NO            YES
          │              │
          X              ↓
       SKIP         ┌──────────────────────┐
                │  FILTER PASSED ✓        │
                │  Proceed to place orders│
                └──────┬───────────────────┘
                       │
                       ↓
        ┌──────────────────────────────────┐
        │  FOR EACH STOCK:                 │
        │  1. Get LTP (api.get_quotes)    │
        │  2. Calculate qty = 20000/LTP   │
        │  3. Place SELL order            │
        └──────┬───────────────────────────┘
               │
        ┌──────┴──────────────────┐
        ↓                         ↓
      ORDER_ID?              NO ORDER
      EXISTS                     │
        │                        X
        │                     SKIP
        ↓
    ┌───────────────────────┐
    │ Add to active_trades  │
    │ {                     │
    │   order_id: "12345"  │
    │   symbol: "SEPOWER"  │
    │   entry_price: 3.75  │
    │   qty: 5333          │
    │   tp_price: 3.72     │
    │   sl_price: 3.78     │
    │   status: ACTIVE     │
    │ }                     │
    └─────────┬─────────────┘
              │
              ↓
    ┌──────────────────────────────┐
    │  MONITOR THREAD (Every 30s)  │
    │  Checks ALL active_trades    │
    └─────────┬────────────────────┘
              │
              ↓
    ┌────────────────────────────┐
    │ Get Current LTP            │
    │ current_ltp = 3.70         │
    └──────┬─────────────────────┘
           │
        ┌──┴──┬────────┬──────────┐
        ↓     ↓        ↓          ↓
    3.70≤  3.70    3.70     3.70≥
    3.72?  <3.72   between   3.78?
      │       │      │        │
     YES     NO     NO       YES
      │       │      │        │
      ↓       ↓      ↓        ↓
    ✅TP   Stay   Stay      ❌SL
    HIT    Wait   Wait      HIT
      │       │      │        │
      ↓       ↓      │        ↓
    BUY    Print   │       BUY
    TO     P&L    │       TO
    CLOSE   │     │      CLOSE
      │     │     │        │
      ├─────┴─────┴────────┤
      │                    │
      └────┬───────────────┘
           ↓
    ┌─────────────────────┐
    │ Mark CLOSED         │
    │ Log exit_price      │
    │ Log P&L             │
    │ Remove from dict    │
    └─────────────────────┘
              ↓
    ┌──────────────────────┐
    │ Wait 30 seconds      │
    │ Loop again           │
    └──────────────────────┘
```

---

## Key Points

| Component | Details |
|-----------|---------|
| **Webhook Entry** | Receives stock alerts from Chartink |
| **Filter Step** | Time window + stock count validation |
| **Order Placement** | SELL orders placed immediately (no verification) |
| **Active Trades** | Stored in dictionary with order_id as key |
| **Monitoring** | Background thread checks every 30 seconds |
| **Exit Trigger** | Price ≤ TP or Price ≥ SL |
| **Logging** | All trades saved to JSON file |
| **P&L Calculation** | (entry_price - current_ltp) × quantity |

---

## Example Trade Lifecycle

```
TIME: 09:50:00
└─ Webhook received: SEPOWER @ 3.75
└─ Filter passed ✓
└─ Place SELL order at 3.75
└─ active_trades[12345] = {entry_price: 3.75, tp: 3.72, sl: 3.78}
└─ Log: "Trade opened - SEPOWER SELL @ 3.75"

TIME: 09:50:30
└─ Monitor check #1
└─ SEPOWER LTP = 3.74 (within bounds)
└─ No action taken

TIME: 09:51:00
└─ Monitor check #2
└─ SEPOWER LTP = 3.71 ✓ (≤ 3.72 TP)
└─ Place BUY order to close
└─ P&L = (3.75 - 3.71) × 5333 = 213.32 ✓ PROFIT
└─ Trade marked CLOSED
└─ Log: "Trade closed - TP HIT - P&L: +213.32"
```

# Testing on ReqBin - Complete Guide

## Step-by-Step Instructions

### 1️⃣ Start Your Trading System
```
Run the main cell in notebook
✓ See: "✅ Login Successful!"
✓ See: "🚀 TRADING SYSTEM STARTED - SIMPLIFIED"
✓ See: "✅ Monitor thread started"
```

### 2️⃣ Start ngrok
Open PowerShell terminal and run:
```powershell
ngrok http 5000
```

**Copy the ngrok HTTPS URL** (looks like: `https://abcd1234-ngrok-free.app`)

---

### 3️⃣ Go to ReqBin
- Open browser: **https://reqbin.com**
- Click on "Composer" tab

---

### 4️⃣ Setup Request

**Method:** SELECT `POST`

**URL:** Paste your ngrok URL + `/webhook`
```
https://YOUR_NGROK_URL/webhook
```
Example:
```
https://abcd1234-ngrok-free.app/webhook
```

**Headers:** Click "Add Header"
- Key: `Content-Type`
- Value: `application/json`

---

### 5️⃣ Paste JSON Payload

Click on **Body** tab, select **raw** → paste this:

```json
{
    "stocks": "SEPOWER,ASTEC,EDUCOMP,KSERASERA,IOLCP,GUJAPOLLO,EMCO",
    "trigger_prices": "3.75,541.8,2.1,0.2,329.6,166.8,1.25",
    "triggered_at": "2:34 pm",
    "scan_name": "Short term breakouts",
    "scan_url": "short-term-breakouts",
    "alert_name": "Alert for Short term breakouts",
    "webhook_url": "http://your-web-hook-url.com"
}
```

---

### 6️⃣ Send Request
Click **"Send"** button

---

## What Happens Next

### ✅ Expected Response (ReqBin)
```json
{
    "status": "success",
    "received": 7,
    "placed": 0
}
```

**Interpretation:**
- `received: 7` = 7 stocks received
- `placed: 0` = 0 orders placed (because time 14:34 is NOT in valid window!)

### ❌ Check Your Notebook Output
You'll see in the notebook:
```
═════════════════════════════════════════════════════════════════
📡 WEBHOOK RECEIVED - 2026-05-04 14:30:15
═════════════════════════════════════════════════════════════════
Raw payload:
{
    "stocks": "SEPOWER,ASTEC,EDUCOMP,KSERASERA,IOLCP,GUJAPOLLO,EMCO",
    ...
}

Parsed stocks:
[
    {"symbol": "SEPOWER", "trigger_price": 3.75, "time": "14:34", ...},
    ...
]

⏭️  Received 7 stocks (max 3). Skipping.
```

**Why 0 orders?**
- You sent 7 stocks (max is 3) → **FILTERED OUT**
- Time 14:34 (2:34 PM) is NOT in valid windows:
  - Valid: 09:45-10:00, 10:15-10:30, 10:30-10:45, 10:45-11:00, 13:00-13:15
  - Your time: 14:34 (2:34 PM) ❌

---

## Test Cases to Try

### Test 1: Valid Time + Valid Count ✅

Change `triggered_at` to `"9:50 am"` and `stocks` to 3 stocks:

```json
{
    "stocks": "SEPOWER,ASTEC,EDUCOMP",
    "trigger_prices": "3.75,541.8,2.1",
    "triggered_at": "9:50 am",
    "scan_name": "Short term breakouts",
    "scan_url": "short-term-breakouts",
    "alert_name": "Alert for Short term breakouts",
    "webhook_url": "http://your-web-hook-url.com"
}
```

**Expected Result:** `placed: 3` ✅ (All 3 orders placed!)

---

### Test 2: Valid Time + Too Many Stocks ❌

Keep 3 stocks limit:

```json
{
    "stocks": "SEPOWER,ASTEC,EDUCOMP",
    "trigger_prices": "3.75,541.8,2.1",
    "triggered_at": "10:20 am",
    "scan_name": "Short term breakouts",
    "scan_url": "short-term-breakouts",
    "alert_name": "Alert for Short term breakouts",
    "webhook_url": "http://your-web-hook-url.com"
}
```

**Expected Result:** `placed: 3` ✅ (All 3 orders placed!)

---

### Test 3: Valid Time + Too Many Stocks ❌

Try with 5 stocks:

```json
{
    "stocks": "SEPOWER,ASTEC,EDUCOMP,KSERASERA,IOLCP",
    "trigger_prices": "3.75,541.8,2.1,0.2,329.6",
    "triggered_at": "10:20 am",
    "scan_name": "Short term breakouts",
    "scan_url": "short-term-breakouts",
    "alert_name": "Alert for Short term breakouts",
    "webhook_url": "http://your-web-hook-url.com"
}
```

**Expected Result:** `placed: 0` ❌ (Too many stocks, skipped entire webhook!)

---

## Monitoring Output

After successful order placement, check notebook for monitor output every 30 seconds:

```
════════════════════════════════════════════════════════════════
📊 MONITOR CHECK #1 - 2026-05-04 09:50:30
Active trades: 3
════════════════════════════════════════════════════════════════

SEPOWER-EQ | Entry: 3.75 | Current: 3.74
  TP: 3.72 | SL: 3.78
  📈 P&L: -0.05

ASTEC-EQ | Entry: 541.80 | Current: 541.75
  TP: 537.45 | SL: 544.45
  📈 P&L: -0.05

EDUCOMP-EQ | Entry: 2.10 | Current: 2.09
  TP: 2.08 | SL: 2.11
  📈 P&L: -0.01
```

---

## Check Logs

Your trades are saved here:
```
D:\AlgoRepo\ShoonyaAPI_Code\Testing_Use\Execution\session_YYYYMMDD_HHMMSS.json
```

Look for `webhooks` and `trades` arrays with all entry/exit data.

---

## Quick Checklist

- [ ] Notebook running (✅ Login Successful)
- [ ] Monitor thread started (✅ Monitor thread started)
- [ ] ngrok running (check terminal for URL)
- [ ] ReqBin URL = ngrok_url + `/webhook`
- [ ] Headers = Content-Type: application/json
- [ ] JSON payload pasted correctly
- [ ] Valid time in payload (09:45-10:00, etc)
- [ ] Max 3 stocks in payload
- [ ] Click Send
- [ ] Check response & notebook output